In [1]:
# --- Configuración de entorno ---

# Añade el directorio raíz al path para que Python encuentre tus módulos
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))  # Sube dos niveles hasta /src

# Importa configuraciones y librerías globales
from config import *
from utils import *

# Carga de los ficheros de datos parciales

Se cargan todos los ficheros de servicios que se han ido creando en los notebooks parciales

In [2]:
def load_with_cusec(path):
    df = pd.read_csv(
        path,
        encoding="utf-8-sig",
        sep=";",          # column separator
        decimal=",",      # European decimal format
        dtype=str         # force all fields to string to avoid type issues
    )
    
    # Normalize CUSEC if present
    if "CUSEC" in df.columns:
        df["CUSEC"] = (
            df["CUSEC"]
            .astype(str)
            .str.strip()
            .str.zfill(10)
        )

    # Normalize CMuni if present
    if "CMuni" in df.columns:
        df["CMuni"] = (
            df["CMuni"]
            .astype(str)
            .str.strip()
            .str.zfill(5)
        )
    
    return df
    
# Por sección censal
actividad_seccion = load_with_cusec(os.path.join(DATA_OUTPUTS_DS, "actividad_por_sexo_por_seccion.csv"))
estudios_seccion = load_with_cusec(os.path.join(DATA_OUTPUTS_DS, "estudios_por_sexo_por_seccion.csv"))
fuenteIngresos_seccion = load_with_cusec(os.path.join(DATA_OUTPUTS_DS, "fuente_ingresos_por_seccion.csv"))
gini_seccion = load_with_cusec(os.path.join(DATA_OUTPUTS_DS, "Gini_P20P80_por_seccion.csv"))
rentas_seccion = load_with_cusec(os.path.join(DATA_OUTPUTS_DS, "rentas_por_seccion.csv"))

# Cargo el shp para obtener el id:
ruta_shp_secciones = os.path.join(DATA_OUTPUTS_DIR, "Shapefiles", "cyl_2022.shp")
gdf_secciones = gpd.read_file(ruta_shp_secciones)
gdf_secciones["CUSEC"] = (
    gdf_secciones["CUSEC"]
    .astype(str)
    .str.strip()
    .str.zfill(10)  # rellena con ceros a la izquierda hasta 10 caracteres
)

In [6]:
DS_seccion = (
    actividad_seccion
    .merge(estudios_seccion, on=["Provincia","CUSEC","CMuni","Periodo"], how="inner")
    .merge(fuenteIngresos_seccion, on=["Provincia", "CUSEC","CMuni", "Periodo"], how="inner")
    .merge(gini_seccion, on=["Provincia", "CUSEC","CMuni", "Periodo"], how="inner")
    .merge(rentas_seccion, on=["Provincia", "CUSEC","CMuni", "Periodo"], how="inner")
    .merge(
        gdf_secciones.drop(columns=["geometry","NMUN","NPRO"]),
        on="CUSEC",
        how="left"
    )
)

DS_seccion["Seccion_id"] = DS_seccion["Seccion_id"].astype("Int64")

print(DS_seccion.info())
DS_seccion.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8382 entries, 0 to 8381
Data columns (total 45 columns):
 #   Column                                                                                Non-Null Count  Dtype 
---  ------                                                                                --------------  ----- 
 0   Provincia                                                                             8382 non-null   object
 1   CMuni                                                                                 8382 non-null   object
 2   CUSEC                                                                                 8382 non-null   object
 3   Periodo                                                                               8382 non-null   object
 4   Hombres_Estudiante                                                                    8382 non-null   object
 5   Hombres_Ocupado_a                                                                     8382

,Provincia,CMuni,CUSEC,Periodo,Hombres_Estudiante,Hombres_Ocupado_a,Hombres_Otra_situación_de_inactividad,Hombres_Parado_a,"Hombres_Perceptor_a_pensión_de_incapacidad,_jubilación,_prejubilación",Mujeres_Estudiante,...,Fuente_de_ingreso_salario,Distribución_de_la_renta_P80_P20,Índice_de_Gini,Media_de_la_renta_por_unidad_de_consumo,Mediana_de_la_renta_por_unidad_de_consumo,Renta_bruta_media_por_hogar,Renta_bruta_media_por_persona,Renta_neta_media_por_hogar,Renta_neta_media_por_persona,Seccion_id
3570,Palencia,34120,3412007006,2023,16,148,26,12,121,21,...,"44,9","2,4","25,5",23564,22750,45622,20295,37834,16831,1451
8135,Zamora,49227,4922701001,2023,7,108,7,8,43,7,...,"65,2",2,24,20109,19250,39104,16118,33258,13709,3377
4159,Salamanca,37140,3714001001,2021,21,255,119,36,128,27,...,"56,8","3,2","36,9",16510,13650,36311,12667,30393,10602,1721
7601,Zamora,49021,4902102003,2022,48,378,46,35,160,62,...,"58,9","2,4","30,4",17376,15750,32898,14170,27951,12039,3180
121,Avila,05019,0501905002,2023,47,331,64,35,219,51,...,"50,7",3,"32,1",23379,21350,45828,20658,37557,16930,42


In [8]:
# Creamos la carpeta si no existe
os.makedirs(DATA_OUTPUTS_DS, exist_ok=True)

# Rutas de salida
ruta_seccion = os.path.join(DATA_OUTPUTS_DS, "DS_seccion.csv")

# Guardar DataFrames
DS_seccion.to_csv(
    ruta_seccion,
    index=False,
    encoding="utf-8-sig",
    sep=";",          # separador de columnas compatible con Excel español
    decimal=",",      # separador decimal europeo
    float_format="%.3f"
)

print(f"✅ Archivos guardados correctamente en: {DATA_OUTPUTS_DD}")

✅ Archivos guardados correctamente en: D:\MASTER EN CIENCIA DE DATOS\TFM\TrabajoFinal\ivst-tfm\data\outputs\DD_Dim_demografica
